# splatreg quickstart: register and merge Gaussian splats

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Archerkattri/splatreg/blob/main/examples/splatreg_quickstart.ipynb)

**splatreg** finds the rigid (SE(3)) or similarity (Sim(3), +scale) transform that aligns two
3D Gaussian Splatting scans, then merges and dedupes them into one splat. Pure PyTorch, CPU is fine.

This notebook runs **top-to-bottom on CPU with no external assets**:

1. install splatreg,
2. generate a synthetic demo splat pair with a *known* ground-truth offset,
3. `register` the pair and check the recovered transform against the ground truth,
4. `merge` them into one deduped splat,
5. plot before/after.

Docs: <https://archerkattri.github.io/splatreg/> · Repo: <https://github.com/Archerkattri/splatreg>

In [ ]:
# 1. Install (a few seconds on Colab; torch + numpy are the only dependencies)
try:
    import splatreg
except ImportError:
    %pip install -q splatreg
    import splatreg

import torch

print("splatreg", splatreg.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2. A demo splat pair with known ground truth

We use the repo's own synthetic-object generator (`examples/_example_utils.py`), an anisotropic
shell + lobe object whose pose is fully observable, and offset a copy of it by a **known**
Sim(3): a 25° rotation, a translation, and a 1.15× scale. That known transform is the ground
truth the registration has to recover.

> **Using your own scans instead:** skip this cell and point `target` / `source` at any two
> standard 3DGS `.ply` exports (gsplat, Nerfstudio splatfacto, INRIA, SuperSplat) via
> `splatreg.io.load_ply("scan.ply")`. A small real release-asset pair would slot in here the
> same way (download the two PLYs, then `load_ply` each).

In [ ]:
# Fetch the generator: import locally when running inside the repo, else pull it from GitHub.
import os, urllib.request

if not os.path.exists("_example_utils.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Archerkattri/splatreg/main/examples/_example_utils.py",
        "_example_utils.py",
    )

from _example_utils import make_object_splat, axis_angle_R, sim3_matrix, rot_angle_deg, chamfer_mm

torch.manual_seed(0)

target = make_object_splat(1200, seed=0)                      # the reference scan (stays fixed)

# Ground-truth offset: 25 deg rotation + translation + 1.15x scale
R_gt = axis_angle_R([0.3, 1.0, 0.2], 25.0)
T_gt = sim3_matrix(1.15, R_gt, torch.tensor([0.08, -0.05, 0.03]))
source = make_object_splat.apply_to(target, T_gt)             # the scan to align

print(f"target: {len(target)} Gaussians | source: {len(source)} Gaussians")
print(f"chamfer source -> target BEFORE registration: {chamfer_mm(source.means, target.means):.1f} mm")

## 3. Register

`register(target, source)` returns the 4×4 transform mapping **source → target**. With
`transform="sim3"` the scale is recovered too (splatreg is the only splat registrar that does
this). The default `init="fast"` suits objects / full-overlap captures; see the
[init-modes guide](https://archerkattri.github.io/splatreg/init-modes/) for real-scan options.

In [ ]:
from splatreg import register

result = register(target, source, transform="sim3", quality="balanced")

print("recovered T (maps source -> target):")
print(result.T.numpy().round(4))
print(f"recovered scale : {result.scale:.4f}   (ground truth: {1/1.15:.4f}, the inverse of the 1.15x we applied)")
print(f"rmse            : {result.info.get('rmse', float('nan')):.2e}")
print(f"converged       : {result.converged}")

In [ ]:
# Check against the known ground truth: result.T should invert T_gt.
R_rec = result.T[:3, :3] / result.scale            # de-scale the Sim(3) block -> pure rotation
rot_err = rot_angle_deg(R_rec, R_gt.T)             # vs the inverse ground-truth rotation
scale_err = abs(result.scale * 1.15 - 1.0) * 100   # recovered * applied should be 1

print(f"rotation error vs ground truth : {rot_err:.3f} deg")
print(f"scale error                    : {scale_err:.2f} %")
assert rot_err < 2.0 and scale_err < 2.0, "registration did not recover the ground truth"
print("ground truth recovered.")

## 4. Merge + dedupe

`merge` is **not a naive concat**: it registers every splat onto the reference, bakes the
recovered transform into means/quats/scales, concatenates, then collapses the double-density
overlap with a voxel dedupe.

In [ ]:
from splatreg import merge

fused = merge([target, source], init="fast", quality="balanced")
print(f"merge: {len(target)} + {len(source)} = {len(target)+len(source)} -> {len(fused)} Gaussians after dedupe")

# Aligned copy of the source (what `splatreg align` writes), for the plot below:
from splatreg.api import apply_transform
aligned = apply_transform(source, result.T, result.scale)
print(f"chamfer aligned -> target AFTER registration: {chamfer_mm(aligned.means, target.means):.2f} mm")

## 5. Before / after

In [ ]:
import matplotlib.pyplot as plt

def scatter(ax, g, color, label):
    p = g.means.detach().cpu().numpy()
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=2, alpha=0.5, c=color, label=label)

fig = plt.figure(figsize=(13, 5))
for i, (b, title) in enumerate([(source, "before: raw scans"), (aligned, "after: splatreg align")]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    scatter(ax, target, "#17becf", "target")
    scatter(ax, b, "#ff6b5b", "source")
    ax.set_title(title)
    ax.set_box_aspect((1, 1, 1))
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 6. Save, and the same thing from the shell

The fused splat is a standard 3DGS PLY: it opens directly in
[SuperSplat](https://superspl.at/editor), gsplat, Nerfstudio, or any 3DGS viewer.

In [ ]:
from splatreg.io import save_ply, load_ply

save_ply(target, "target.ply")
save_ply(source, "source.ply")
save_ply(fused, "fused.ply")
print("wrote target.ply / source.ply / fused.ply:", len(load_ply("fused.ply")), "Gaussians reload OK")

In [ ]:
# pip install also gives you a CLI: the identical workflow with no Python:
!splatreg align target.ply source.ply -o aligned.ply --transform sim3 --quality balanced --device cpu
!splatreg info fused.ply

## Where next

- **Quickstart / API docs:** <https://archerkattri.github.io/splatreg/>
- **PLY interop** (splatfacto / INRIA / SuperSplat round-trip, SH-under-rotation detail):
  <https://archerkattri.github.io/splatreg/ply-interop/>
- **Benchmarks** (official 3DMatch 91.5%, real-splat merge 5.1× Chamfer win, honest limits):
  <https://archerkattri.github.io/splatreg/benchmarks/>
- Object pose (`estimate_object_pose`, ADD/ADD-S) and camera localization
  (`localize_camera`) are in the same package; see the quickstart.

**Know the edges:** overlap ≤ 40% is genuinely ambiguous (flagged via
`result.info["ambiguous"]`, never silently wrong); scale is unobservable under ~20% overlap.
`merge` is designed for high-overlap captures.